In [88]:
import pandas as pd
import numpy as np

In [89]:
orders_cols = ['id', 'status', 'placed_at']
order_items_cols = ['id','order_id','product_variant_id','quantity']
products_cols = ['id','name']
product_variants_cols = ['id','product_id','is_active']

In [90]:
orders = pd.read_csv("./dados/orders.csv", usecols=orders_cols)
order_items = pd.read_csv("./dados/order_items.csv", usecols=order_items_cols)
products = pd.read_csv("./dados/products.csv", usecols=products_cols)
product_variants = pd.read_csv("./dados/product_variants.csv", usecols=product_variants_cols)

### Operação de merge dos datasets

In [91]:
orders.shape

(48998, 3)

In [92]:
orders.rename(columns={'id': 'order_id'}, inplace=True)
orders_merge = pd.merge(orders, order_items, on="order_id", how="outer")
orders_merge.shape

(147320, 6)

In [93]:
orders_merge.head()

,order_id,status,placed_at,id,product_variant_id,quantity
0,1,paid,2022-09-06 05:37:37,1,113,6
1,2,paid,2023-02-03 04:36:21,2,293,9
2,2,paid,2023-02-03 04:36:21,3,366,2
3,2,paid,2023-02-03 04:36:21,4,561,1
4,2,paid,2023-02-03 04:36:21,5,385,5


In [94]:
products.shape

(500, 2)

In [95]:
product_variants.shape

(1009, 3)

In [96]:
products.rename(columns={'id': 'product_id'}, inplace=True)
product_variants.rename(columns={'id': 'product_variant_id'}, inplace=True)
product_merge = pd.merge(products, product_variants, on="product_id", how="outer")
product_merge.shape

(1009, 4)

In [97]:
product_merge.head()

,product_id,name,product_variant_id,is_active
0,1,Motor de Popa 6014,1,True
1,1,Motor de Popa 6014,2,True
2,1,Motor de Popa 6014,3,True
3,2,Vela Mestra 3870,4,True
4,2,Vela Mestra 3870,5,True


In [98]:
df = pd.merge(orders_merge, product_merge, on="product_variant_id", how="outer")
df.shape

(147320, 9)

In [99]:
df.head()

,order_id,status,placed_at,id,product_variant_id,quantity,product_id,name,is_active
0,28,paid,2026-12-25 03:59:07,85,1,7,1,Motor de Popa 6014,True
1,181,confirmed,2026-12-02 14:16:17,561,1,7,1,Motor de Popa 6014,True
2,380,paid,2025-05-18 06:07:52,1175,1,7,1,Motor de Popa 6014,True
3,437,paid,2025-12-08 00:03:25,1347,1,9,1,Motor de Popa 6014,True
4,707,confirmed,2025-04-23 03:34:47,2189,1,1,1,Motor de Popa 6014,True


In [100]:
# Transformar tipo do atributo placed_at em datetime para facilitar manipulação posterior
df['placed_at'] = pd.to_datetime(df['placed_at'])
df.dtypes

order_id                       int64
status                        object
placed_at             datetime64[ns]
id                             int64
product_variant_id             int64
quantity                       int64
product_id                     int64
name                          object
is_active                       bool
dtype: object

### Filtragem de dados e divisão de treino e teste
Premissas obrigatórias:
- O período de treino deve incluir dados até 31/12/2025.
- O período de teste deve ser o primeiro trimestre de 2026.
- A previsão deve ser feita em base mensal.
- Considere apenas o produto: "Bússola de Bordo 702"

In [101]:
np.unique(df["status"])

array(['cancelled', 'confirmed', 'draft', 'paid'], dtype=object)

In [102]:
# Filtrar apenas pedidos com status de 'confirmado' ou 'pago'
df_filtrado = df[df["status"].isin(['confirmed', 'paid'])]
df_filtrado.shape

(125496, 9)

In [103]:
# Filtrar especificamente por pedidos com o produto Bússola de Bordo 702
df_filtrado = df_filtrado[df_filtrado["name"] == "Bússola de Bordo 702"]
df_filtrado.shape

(408, 9)

In [118]:
# Criar coluna Mês e Ano
df_filtrado["month"] = df_filtrado["placed_at"].dt.month
df_filtrado["year"] = df_filtrado["placed_at"].dt.year
df_filtrado.columns

Index(['order_id', 'status', 'placed_at', 'id', 'product_variant_id',
       'quantity', 'product_id', 'name', 'is_active', 'month', 'year'],
      dtype='object')

In [105]:
# Teste: Agrupando por trimestre na coluna 'placed_at' e calculando o total de registros por trimestre
resultado = df_filtrado.groupby(pd.Grouper(key='placed_at', freq='ME')).size()
resultado

placed_at
2020-01-31     5
2020-02-29     3
2020-03-31     3
2020-04-30     4
2020-05-31     2
              ..
2026-08-31     5
2026-09-30     9
2026-10-31     4
2026-11-30     6
2026-12-31    12
Freq: ME, Length: 84, dtype: int64

In [106]:
# Teste: Agrupando por mês na coluna 'placed_at' e calculando a quantidade de pedidos mensal
df_filtrado.groupby(pd.Grouper(key='placed_at', freq='ME'))['quantity'].sum()

placed_at
2020-01-31    29
2020-02-29    16
2020-03-31    17
2020-04-30    28
2020-05-31     5
              ..
2026-08-31    25
2026-09-30    46
2026-10-31    20
2026-11-30    35
2026-12-31    64
Freq: ME, Name: quantity, Length: 84, dtype: int64

In [119]:
# Criar novo dataframe somando os valores da coluna "quantity" respeitando mês e ano de "placed_at"
monthly_summary_df = (
    df_filtrado.groupby(df_filtrado['placed_at'].dt.to_period('M'))[['quantity']]
    .sum()
    .reset_index()
)
monthly_summary_df

,placed_at,quantity
0,2020-01,29
1,2020-02,16
2,2020-03,17
3,2020-04,28
4,2020-05,5
...,...,...
74,2026-08,25
75,2026-09,46
76,2026-10,20
77,2026-11,35


In [108]:
# Conjunto de treino deve incluir dados até 31/12/2025
df_treino = monthly_summary_df[monthly_summary_df["placed_at"] < "2026-01"]
df_treino.shape

(67, 2)

In [109]:
# Conjunto de teste abrange entradas/pedidos apenas do 1° trimestre de 2026
df_teste = monthly_summary_df[(monthly_summary_df["placed_at"] >= "2026-01") & (monthly_summary_df["placed_at"] < "2026-04")]
df_teste.shape

(3, 2)

### Estimativa do modelo e cálculo da MAE

In [113]:
from statsmodels.tsa.arima.model import ARIMA

# Exemplo de ajuste de um modelo ARIMA/MA
# y_t = c + e_t + theta * e_{t-1}
modelo = ARIMA(df_treino['quantity'], order=(0, 0, 3))  # order = (p, d, q) -> q=1 para MA(1)
resultado = modelo.fit()

print(resultado.summary())

# Fazendo previsões para os próximos 5 passos
previsao = resultado.forecast(steps=3)
print(previsao)

                               SARIMAX Results                                
Dep. Variable:               quantity   No. Observations:                   67
Model:                 ARIMA(0, 0, 3)   Log Likelihood                -274.760
Date:                dom, 01 nov 2026   AIC                            559.520
Time:                        02:35:01   BIC                            570.543
Sample:                             0   HQIC                           563.882
                                 - 67                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         25.7541      3.160      8.151      0.000      19.561      31.947
ma.L1          0.2924      0.141      2.072      0.038       0.016       0.569
ma.L2          0.2673      0.141      1.889      0.0

In [114]:
df_teste

,placed_at,quantity
67,2026-01,76
68,2026-02,55
69,2026-03,51


In [115]:
# Compare as previsões com os valores reais do período de teste utilizando a métrica: MAE (Mean Absolute Error)
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(previsao, df_teste["quantity"])
print("MAE:", mae)

MAE: 35.50591265771879


In [116]:
mae = np.mean(np.abs(df_teste["quantity"] - previsao))
print("MAE:", mae)

MAE: 35.50591265771879


### Análise do modelo

5. Responda objetivamente:
     a. O baseline é adequado para esse produto?
     b. Cite uma limitação desse método.

     a. Não. O valor do MAE indica que o modelo tende a subestimar ou superestimar a quantidade de itens vendidos por mês em aproximadamente 35 unidades em média, uma quantidade significativa.

     b. O modelo não consegue capturar sazonalidade, um fator que pode ser importante na avaliação de estoque, e considera apenas o erro dos três meses anteriores para fazer a previsão em vez de avaliar uma tendência mais duradoura / de longo prazo.

Questão 6.3 - Explique:
    a. Como o baseline foi construído?
    b. Como evitou data leakage?
    c. Uma limitação do modelo proposto.

6. Respostas
    
    a. Uni os dados necessários por meio da relação de ID entre os arquivos CSVs. Filtrei os registros para considerar apenas pedidos do item em questão (Bússola de Bordo 702) que foram confirmados ou pagos. Agrupei os dados segundo o mês e ano, criando um sumário da quantidade de vendas por mês/ano para realizar a previsão em base mensal. Dividi os conjuntos de treino e teste respeitando o período estabelecido no enunciado da questão. Por fim, treinei o modelo de médias móveis considerando o erro dos últimos três períodos e estimei a venda do primeiro trimestre de 2026.

    b. Agrupei os dados e somei a quantidade de itens pedidos respeitando os critérios de mês e ano, o que possibilitou distinguir itens vendidos (por exemplo) em janeiro de 2025 de janeiro de 2026 e impedir que esses valores fossem misturados na previsão do modelo. Dividi os conjuntos de treino e teste e conferi se a quantidade de linhas de cada dataframe correspondia ao esperado, assim como a quantia de itens por mês/ano.

    c. Não considera sazonalidade nem tendência de longa duração.